# Species Name Enrichment via GBIF API

Takes a CSV with scientific names, looks each one up in the **GBIF backbone taxonomy**, and outputs an enriched CSV with: `common_name`, `source_db`, `source_url`, and `taxon_id`.

**How it works:**
- Uses `pygbif` to query GBIF's taxonomic name-matching API
- For each matched species, fetches its English vernacular (common) name via GBIF's species endpoint
- Outputs one row per input name — unmatched names come back with blank enrichment columns

**Rate limits:** Adds a 200ms delay between requests to stay within GBIF's fair-use guidelines.

In [ ]:
!pip install pandas pygbif -q

## 1. Upload your species list

Your CSV should have a column named **`scientific_name`** (case-insensitive). Extra columns are ignored — only the scientific names are used for lookup.

Example input:
```csv
scientific_name
Lantana camara
Cymbopogon citratus
Carica papaya
```

If your column has a different name, the notebook will fall back to the first column in the file.

In [ ]:
import pandas as pd
from google.colab import files

def read_csv_safe(path):
    for enc in ['utf-8', 'cp1252', 'latin-1']:
        try: return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError: continue
    return pd.read_csv(path, encoding='utf-8', errors='replace')

uploaded = files.upload()
input_csv = list(uploaded.keys())[0]
print(f'Using: {input_csv}')

df = read_csv_safe(input_csv)
name_col = 'scientific_name' if 'scientific_name' in df.columns else df.columns[0]
unique_names = {str(x).strip().lower(): str(x).strip() for x in df[name_col].dropna()}
print(f'Loaded {len(unique_names)} unique scientific names')

## 2. Enrich via GBIF API

Each scientific name is sent through two GBIF API calls:

1. **`species/match`** — matches the name against the GBIF backbone taxonomy. Returns a `usageKey` (GBIF's internal taxon ID), match confidence, and taxonomic classification.
2. **`species/{key}/vernacularNames`** — fetches common names for the matched taxon. The first English (`eng`) vernacular name is used.

A 200ms delay is added between requests to avoid rate-limiting. Progress is printed every 10 lookups.

In [ ]:
import time
from pygbif import species as gbif_species

OUTPUT = ['scientific_name', 'common_name', 'source_db', 'source_url', 'taxon_id']
rows = []
matched = 0
unmatched_keys = []

for i, (key, orig_name) in enumerate(unique_names.items()):
    try:
        # Step 1: match name against GBIF backbone taxonomy
        match = gbif_species.name_backbone(scientificName=orig_name)
        diag = match.get('diagnostics', {})
        usage = match.get('usage', {})
        if diag.get('matchType') == 'NONE' or not usage.get('key'):
            unmatched_keys.append(key)
            rows.append({c: '' for c in OUTPUT})
            rows[-1]['scientific_name'] = orig_name
            continue

        usage_key = usage['key']

        # Step 2: fetch English vernacular name
        common_name = ''
        try:
            vnames = gbif_species.name_usage(key=usage_key, data='vernacularNames', limit=50)
            for v in vnames.get('results', []):
                # GBIF uses ISO 639-3 (3-letter) language codes, e.g. 'eng' for English
                if v.get('language') == 'eng' and v.get('vernacularName'):
                    common_name = v['vernacularName']
                    break
        except Exception:
            pass

        rows.append({
            'scientific_name': orig_name,
            'common_name': common_name,
            'source_db': 'GBIF',
            'source_url': f'https://www.gbif.org/species/{usage_key}',
            'taxon_id': str(usage_key)
        })
        matched += 1

    except Exception as e:
        unmatched_keys.append(key)
        rows.append({c: '' for c in OUTPUT})
        rows[-1]['scientific_name'] = orig_name

    # Polite delay between requests
    time.sleep(0.2)

    if (i + 1) % 10 == 0 or (i + 1) == len(unique_names):
        print(f'{i + 1}/{len(unique_names)} — {matched} matched', end='\r')

print()
unmatched = len(unique_names) - matched
print(f'Matched: {matched}, Unmatched: {unmatched}')
if unmatched_keys:
    print(f'Unmatched: {unmatched_keys[:20]}')
    if len(unmatched_keys) > 20:
        print(f'  ... and {len(unmatched_keys) - 20} more')

## 3. Download enriched CSV

The output file `species_enriched.csv` contains these columns:

| Column | Description |
|---|---|
| `scientific_name` | Original name from your input (preserved as-is) |
| `common_name` | English vernacular name from GBIF (blank if none found) |
| `source_db` | Always `GBIF` for matched rows |
| `source_url` | Link to the species page on GBIF.org |
| `taxon_id` | GBIF usage key (numeric ID) |

Unmatched names have blank values in all enrichment columns.

In [ ]:
res_df = pd.DataFrame(rows, columns=OUTPUT)

out_name = 'species_enriched.csv'
res_df.to_csv(out_name, index=False)
print(f'Saved: {out_name}')
print(f'Total: {len(res_df)}, Matched: {matched}')
files.download(out_name)